In [1]:
import os
import json
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
system_message = """You are a helpful assistant for an Airline called FLightAI. 
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
 """

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [
        {"role":"system", "content": system_message} + history + {"role":"user", "content":message}
    ]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# Add Tool

In [ ]:
ticket_prices = {"London":"$799", "Paris":"$501", "Tokyo":"$1500"}

def get_ticket_estimate(destination_city):
    print(f"Tool called for city: {destination_city}")
    price = ticket_prices.get(destination_city, "Unknown City")
    return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
get_ticket_estimate("Paris")

Tool called for city: Paris


'The price of a ticket to Paris is $501'

## Define the JSON structure for the tool

In [ ]:
price_function = {
    "name":"get_ticket_estimate",
    "description":"Get the price of a return ticket to the destination city.",
    "parameters":{
        "type":"object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description":"The city the customer wants to travel to."
            }
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [ ]:
tools = [{"type":"function", "function":price_function}]

In [ ]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_estimate',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city the customer wants to travel to.'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [ ]:
json.dumps(tools)

'[{"type": "function", "function": {"name": "get_ticket_estimate", "description": "Get the price of a return ticket to the destination city.", "parameters": {"type": "object", "properties": {"destination_city": {"type": "string", "description": "The city the customer wants to travel to."}}, "required": ["destination_city"], "additionalProperties": false}}}]'

## Use the tool now

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_estimate":
        arguments = json.loads(tool_call.function.arguments)
        print("Arguments: ", arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_estimate(city)
        response = {
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}]\
                     + history\
                         + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Arguments:  {'destination_city': 'Paris'}
Tool called for city: Paris


# Add Multiple Tool Calls

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_estimate":
            arguments = json.loads(tool_call.function.arguments)
            print("Arguments: ", arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_estimate(city)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}]\
                     + history\
                         + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction 